In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
import pandas as pd
import numpy as np # Used by pandas internally, but good to keep if needed
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
# import matplotlib.pyplot as plt # Not needed since Plotly Express is used

#### FIX ME #####
# Correct import to use your CRUD Python module file name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "cs-340" # Updated password from user's input

# Connect to database via CRUD Module
try:
    db = AnimalShelter(username, password)
except Exception as e:
    print(f"FATAL ERROR: Could not connect to MongoDB. Details: {e}")
    db = None
    
# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
if db:
    data_list = db.read({})
    df = pd.DataFrame.from_records(data_list if data_list else [])
else:
    df = pd.DataFrame()

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here.
if '_id' in df.columns:
    df.drop(columns=['_id'],inplace=True)
    
# Data Cleaning for Geolocation Chart (Ensuring coordinates are numeric and not NaN)
if 'location_long' in df.columns and 'location_lat' in df.columns:
    df['location_long'] = pd.to_numeric(df['location_long'], errors='coerce')
    df['location_lat'] = pd.to_numeric(df['location_lat'], errors='coerce')
    # Filter out records where coordinates are null or zero
    df = df.dropna(subset=['location_long', 'location_lat'])
    df = df[(df['location_long'] != 0) | (df['location_lat'] != 0)]


# --- Logo Encoding and Unique Identifier (Integrated Fix) ---
image_filename = 'GraziosoSalvareLogo.png'
try:
    with open(image_filename, 'rb') as f:
        encoded_image = base64.b64encode(f.read()).decode('ascii')
    logo_src = 'data:image/png;base64,{}'.format(encoded_image)
except FileNotFoundError:
    print(f"WARNING: Logo file '{image_filename}' not found. Using placeholder.")
    logo_src = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNk+M9QDwADhgGAWjRDAAAAAElFTkSuQmCC"

# --- Filter Options Setup (Based on Dashboard Specifications) ---
filter_options = [
    {'label': 'Water Rescue (DOG - Mixed)', 'value': 'water_rescue'},
    {'label': 'Mountain/Wilderness Rescue (DOG - German Shepherd)', 'value': 'mountain_rescue'},
    {'label': 'Disaster/Individual Rescue (DOG - Doberman)', 'value': 'disaster_rescue'},
    {'label': 'All/No Filter', 'value': 'all'}
]

# --- Map Update Helper Function ---
def create_leaflet_map(viewData, index):    
    if not viewData or len(viewData) == 0:
        return [
            dl.Map(style={'width': '100%', 'height': '500px'},
                   center=[30.75,-97.48], zoom=10, 
                   children=[dl.TileLayer(id="base-layer-id")]
            )
        ]
        
    dff = pd.DataFrame.from_dict(viewData)
    
    row = index[0] if index and index[0] < len(dff) else 0
        
    selected_row = dff.iloc[row]
    
    # Use column names for robustness
    lat = selected_row['location_lat']
    lon = selected_row['location_long']
    breed = selected_row['breed']
    name = selected_row['name']

    return [
        dl.Map(style={'width': '100%', 'height': '500px'},
           center=[lat, lon], zoom=10, children=[
           dl.TileLayer(id="base-layer-id"),
           dl.Marker(position=[lat, lon],
              children=[
              dl.Tooltip(breed),
              dl.Popup([
                 html.H1("Animal Name"),
                 html.P(name)
             ])
          ])
       ])
    ]


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

app.layout = html.Div([
    
    # --- HEADER SECTION (Logo and Identifier) ---
    html.Div([
        # Logo
        html.Img(src=logo_src, style={'height': '100px', 'float': 'left', 'margin-right': '20px'}),
        # Unique Identifier
        html.Center(html.B(html.H1('Grazioso Salvare Rescue Dashboard - Bruno DeSousa', style={'color': '#1E90FF'}))),
    ], style={'padding': '10px', 'borderBottom': '2px solid #1E90FF', 'display': 'flex', 'alignItems': 'center'}),
    
    html.Hr(),
    
    # --- INTERACTIVE FILTER CONTROLS ---
    html.Div([
        html.H3("Select Rescue Type Filter:"),
        dcc.RadioItems(
            id='filter-type',
            options=filter_options,
            value='all', # Default value
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        ),
    ], style={'padding': '20px', 'backgroundColor': '#f8f8f8'}),
    
    html.Hr(),
    
    # --- DATA TABLE WIDGET ---
    dash_table.DataTable(id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_current=0,
        page_size=10,           # Pagination
        sort_action="native",   # Sorting
        filter_action="native", # Filtering
        row_selectable="single", # Required for map interaction
        selected_rows=[0],      # Select first row by default
        style_header={'backgroundColor': 'lightgrey', 'fontWeight': 'bold'},
        style_data_conditional=[
            { 'if': {'row_index': 'odd'}, 'backgroundColor': 'rgb(248, 248, 248)' },
        ],
        virtualization=True
    ),
    
    html.Br(),
    html.Hr(),
    
    # This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
        style={'display' : 'flex', 'flex-wrap': 'wrap'},
        children=[
            # Second Chart Placeholder
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '49%', 'padding': '10px'}
            ),
            # Geolocation Chart Placeholder
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '49%', 'padding': '10px'}
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# --- CALLBACK 1: Filter Control (Controller -> Data Table) ---
@app.callback(
    Output('datatable-id','data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Set up the query based on the selected radio button value
    if filter_type == 'water_rescue':
        # Water Rescue: Dog, Mixed Breed, "Outcome Type: Adoption"
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Mix"},
            "outcome_type": "Adoption"
        }
    elif filter_type == 'mountain_rescue':
        # Mountain/Wilderness: Dog, German Shepherd, "Outcome Type: Transfer"
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "German Shepherd"},
            "outcome_type": "Transfer"
        }
    elif filter_type == 'disaster_rescue':
        # Disaster/Individual: Dog, Doberman, "Outcome Type: Return to Owner"
        query = {
            "animal_type": "Dog",
            "breed": {"$regex": "Doberman"},
            "outcome_type": "Return to Owner"
        }
    elif filter_type == 'all':
        # All/No Filter: Retrieve all data
        query = {}
    else:
        # Default safety net
        query = {}
        
    # Execute the query using the CRUD module with error handling
    if db:
        try:
            filtered_data_list = db.read(query)
            
            # --- CRITICAL FIX: Remove MongoDB's ObjectId for JSON serialization ---
            cleaned_data = []
            if filtered_data_list:
                for record in filtered_data_list:
                    record.pop('_id', None) # Safely remove the non-serializable '_id' field
                    cleaned_data.append(record)
                return cleaned_data
            # ----------------------------------------------------------------------
            
            return []
        except Exception as e:
            # Print the error to your console for debugging, and return an empty list
            print(f"ERROR: Failed to read data from MongoDB with query {query}. Details: {e}")
            return []
    else:
        return []

# --- CALLBACK 2: Data Table Highlighting (Row Styling) ---
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_rows')]
)
def update_styles(selected_rows):
    # Base style for odd rows (fixed element)
    styles = [
        {
            'if': {'row_index': 'odd'},
            'backgroundColor': 'rgb(248, 248, 248)'
        }
    ]
    
    # Concatenate the base style with the styles for selected rows (list comprehension)
    if selected_rows:
        styles += [
            {
                'if': {'row_index': i},
                'backgroundColor': '#D2F3FF', # Highlight color
                'fontWeight': 'bold'
            }
            for i in selected_rows
        ]
        
    return styles

# --- CALLBACK 3: Second Chart Update (Bar Chart) ---
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if not viewData:
        # Return an empty graph component if no data is available
        return dcc.Graph(figure=px.bar(title="No data available for charting."))
    
    # Convert visible data to a DataFrame
    dff = pd.DataFrame.from_dict(viewData)
    
    # Bar Chart: Outcome Subtype Distribution
    counts = dff['outcome_subtype'].value_counts().nlargest(10).reset_index()
    counts.columns = ['Outcome Subtype', 'Count']
    
    # Create a Bar Chart using Plotly Express
    fig = px.bar(
        counts,
        x='Outcome Subtype',
        y='Count',
        title='Top 10 Outcome Subtypes (Filtered)',
        color='Outcome Subtype',
        template='plotly_white'
    )
    
    return dcc.Graph(figure=fig)

# --- CALLBACK 4: Geolocation Chart Update (Map View) ---
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "selected_rows")]
)
def update_map(viewData, index):
    # This calls the helper function defined earlier in the script
    return create_leaflet_map(viewData, index)

# Run app and display result in jupyterlab mode
app.run_server()